# Ridge Regression - Scikit-Learn

## What is Ridge Regression?

Ridge Regression is a version of linear regression that adds an L2 penalty to control large coefficient values. While Linear Regression only minimizes prediction error, it can become unstable when features are highly correlated. Ridge solves this by shrinking coefficients making the model more stable and reducing overfitting.

---

## Key Benefits

| Benefit | Description |
|---------|-------------|
| L2 Regularization | Adds an L2 penalty to model weights |
| Bias-Variance Tradeoff | Controls how large coefficients can grow |
| Multicollinearity | Improves stability when features overlap |
| Generalization | Helps the model generalize better on new data |

---

## Ridge Formula

$$\text{Loss} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} w_j^2$$

---

## Selection of the Ridge Parameter

### 1. Cross-Validation
- **K-Fold Cross-Validation:** Dataset divided into K folds
- **Leave-One-Out Cross-Validation (LOOCV):** Each observation acts once as validation point

### 2. Generalized Cross-Validation (GCV)
- Efficient alternative to LOOCV, avoids explicitly splitting data

### 3. Information Criteria
- AIC and BIC balance model fit with complexity

### 4. Empirical Bayes Methods
- Treats k as a Bayesian hyperparameter

### 5. Stability Selection
- Repeatedly fits model on subsampled datasets

---

## Applications

| Domain | Use Case |
|--------|----------|
| Finance | Risk modeling, portfolio optimization |
| Econometrics | Economic forecasting |
| Genomics | Gene expression analysis |
| Marketing Analytics | Customer behavior prediction |

---

## Advantages

- Overfitting Control: Shrinks coefficients to prevent memorizing noise
- Correlation Support: Handles correlated predictors effectively
- Better Generalization: Produces stable predictions on new data
- Feature Retention: Keeps all features in the model

---

## Limitations

- No Feature Selection: Coefficients never reduced to exact zero
- Hyperparameter Sensitivity: Requires careful lambda (alpha) tuning
- Irrelevant Feature Impact: Affected when many inputs add no useful information
- Reduced Interpretability: Heavy shrinkage can obscure true effect of predictors
- Poor Fit for Sparse Models: Not ideal when only few predictors matter

---

## One-Line Summary

**Ridge Regression adds L2 penalty to linear regression, reducing overfitting by shrinking coefficients and handling multicollinearity.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression, RidgeCV
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

print("="*50)
print("RIDGE REGRESSION - SCIKIT-LEARN")
print("="*50)

In [ ]:
# Create synthetic dataset
np.random.seed(0)
n_samples = 200
n_features = 6

X = np.random.randn(n_samples, n_features)
true_coef = np.array([3.2, -1.5, 0.7, 0, 2.8, -0.5])
y = X.dot(true_coef) + np.random.randn(n_samples) * 0.6

print("Dataset created:")
print(f"Samples: {n_samples}")
print(f"Features: {n_features}")
print(f"True coefficients: {true_coef}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"MSE: {lr_mse:.4f}")
print(f"R2 Score: {lr_r2:.4f}")
print(f"Coefficients: {lr.coef_}")

## Ridge Regression with Alpha=1.0

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("Ridge Regression Results (alpha=1.0):")
print(f"MSE: {ridge_mse:.4f}")
print(f"R2 Score: {ridge_r2:.4f}")
print(f"Coefficients: {ridge.coef_}")

## Ridge vs Linear Regression Comparison

In [ ]:
print("\n" + "="*50)
print("Linear Regression vs Ridge Regression")
print("="*50)

print(f"{'Metric':<15} {'Linear':<15} {'Ridge (alpha=1)':<15}")
print("-"*50)
print(f"{'MSE':<15} {lr_mse:<15.4f} {ridge_mse:<15.4f}")
print(f"{'R2 Score':<15} {lr_r2:<15.4f} {ridge_r2:<15.4f}")

## Effect of Different Alpha Values

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 500]
mse_scores = []
coef_magnitudes = []

print("\n" + "="*50)
print("Effect of Alpha on Ridge Regression")
print("="*50)

for alpha in alphas:
    ridge_alpha = Ridge(alpha=alpha)
    ridge_alpha.fit(X_train, y_train)
    y_pred = ridge_alpha.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)
    coef_magnitudes.append(np.sum(np.abs(ridge_alpha.coef_)))
    print(f"Alpha = {alpha:6}: MSE = {mse:.4f}, Sum|Coefficients| = {coef_magnitudes[-1]:.2f}")

## GridSearchCV for Best Alpha

In [ ]:
param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100, 500]}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring="neg_mean_squared_error")
grid.fit(X_train, y_train)

best_ridge = grid.best_estimator_
best_alpha = grid.best_params_["alpha"]
y_pred_best = best_ridge.predict(X_test)

best_mse = mean_squared_error(y_test, y_pred_best)
best_r2 = r2_score(y_test, y_pred_best)

print("\nGridSearchCV Results:")
print(f"Best alpha selected: {best_alpha}")
print(f"MSE (best alpha): {best_mse:.4f}")
print(f"R2 Score (best alpha): {best_r2:.4f}")

## RidgeCV (Built-in Cross-Validation)

In [ ]:
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 500], cv=5)
ridge_cv.fit(X_train, y_train)

print("\nRidgeCV Results:")
print(f"Best alpha (RidgeCV): {ridge_cv.alpha_}")
print(f"MSE (RidgeCV): {mean_squared_error(y_test, ridge_cv.predict(X_test)):.4f}")

## Cross-Validation Scores

In [ ]:
cv_scores = cross_val_score(best_ridge, X_scaled, y, cv=5, scoring='r2')

print("\n" + "="*50)
print("Cross-Validation Results")
print("="*50)
print(f"CV Scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Std CV Score: {cv_scores.std():.4f}")

## Final Model Comparison

In [ ]:
print("\n" + "="*50)
print("Final Model Comparison")
print("="*50)

print(f"{'Model':<20} {'MSE':<12} {'R2 Score':<12}")
print("-"*50)
print(f"{'Linear Regression':<20} {lr_mse:<12.4f} {lr_r2:<12.4f}")
print(f"{'Ridge (alpha=1)':<20} {ridge_mse:<12.4f} {ridge_r2:<12.4f}")
print(f"{'Ridge (best alpha)':<20} {best_mse:<12.4f} {best_r2:<12.4f}")

In [ ]:
# Visualize actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title(f'Ridge Regression: Actual vs Predicted (alpha={best_alpha})')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Day Completed
print("\n" + "="*50)
print("RIDGE REGRESSION - SCIKIT-LEARN COMPLETED")
print("="*50)
print("Topics covered:")
print("- Ridge Regression with Scikit-Learn")
print("- Linear Regression baseline comparison")
print("- Effect of alpha on coefficients")
print("- GridSearchCV for alpha selection")
print("- RidgeCV for built-in cross-validation")
print("- Cross-validation scores")
print("- Actual vs Predicted visualization")
print("="*50)"
print("\n*** RIDGE REGRESSION MODULE COMPLETED ***")